# Web3 Python 100本ノック：第1章
## §1-9：APIと連携し、Gas代を「日本円」で可視化せよ

1-8で現在のGas代が「何 Gwei か」は分かりましたが、私たち日本人にとって「30 Gwei」と言われても高いのか安いのかピンときません。

この問題では、外部のAPI（CoinGecko）から現在のETHの日本円価格を取得し、**「もし今、イーサリアムを送金したら手数料は何円かかるのか？」**を具体的に計算する実用的なスクリプトを作成します。

> **注意**: 各セルを順番に実行してください。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In [ ]:
!pip install web3==7.16.0 requests

## 実装のポイント：送金にかかる「Gas Limit」

ETHを別のアドレスに送るだけの単純な取引の場合、消費されるGasの量（Gas Limit）は **21,000** と厳密に決まっています。
したがって、1回の送金にかかるトータルコストは `現在のGas代 × 21000` で計算できます。

In [4]:
from web3 import Web3
import requests

RPC_URL = "https://eth.drpc.org"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'headers': headers, 'timeout': 5}))

print(" リアルタイム送金コスト（日本円）を計算中...\n")

try:
    # 1. ブロックチェーンから現在のGas代（Wei）を取得
    gas_price_wei = w3.eth.gas_price
    
    # 2. 1回の標準的なETH送金にかかる総コスト（Wei）を計算
    GAS_LIMIT_TRANSFER = 21000
    total_cost_wei = gas_price_wei * GAS_LIMIT_TRANSFER
    
    # 3. Wei を ETH（Ether）に変換
    total_cost_eth = w3.from_wei(total_cost_wei, 'ether')
    
    # 4. CoinGecko APIから、現在の 1 ETH の日本円価格を取得
    api_url = "https://api.coingecko.com/api/v3/simple/price?ids=ethereum&vs_currencies=jpy"
    response = requests.get(api_url, timeout=10)
    eth_jpy_rate = response.json()["ethereum"]["jpy"]
    
    # 5. ETH単位のコストに日本円レートを掛ける
    cost_in_jpy = float(total_cost_eth) * eth_jpy_rate
    
    # 結果の表示
    print(" 計算完了！")
    print("=" * 40)
    print(f"現在のGas代    : {w3.from_wei(gas_price_wei, 'gwei'):.10f} Gwei")
    print(f"現在のETH価格  : 1 ETH = {eth_jpy_rate:,.0f} 円")
    print("-" * 40)
    print(f" 今、ETHを送金した際の手数料: 約 {cost_in_jpy:,.3f} 円")
    print("=" * 40)
    print(" このように、処理を行う前に『日本円でいくらかかるか』をプログラムに計算させることで、手数料が高騰している時の大損を防ぐことができます。")

except Exception as e:
    print(f" エラーが発生しました: {e}")

 リアルタイム送金コスト（日本円）を計算中...

 計算完了！
現在のGas代    : 0.0361738340 Gwei
現在のETH価格  : 1 ETH = 295,216 円
----------------------------------------
 今、ETHを送金した際の手数料: 約 0.224 円
 このように、処理を行う前に『日本円でいくらかかるか』をプログラムに計算させることで、手数料が高騰している時の大損を防ぐことができます。
